In [2]:
# ============================================
# ICU AI SYSTEM — DATA PREPROCESSING
# ============================================

# Core Libraries
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [3]:
# ============================================
# LOAD RAW ICU DATASET
# ============================================

df = pd.read_csv("../data/raw/icu_data.csv")

print("Dataset Loaded Successfully")
print("Shape:", df.shape)

df.head()

Dataset Loaded Successfully
Shape: (104557, 67)


,ID,UHID,IPNumber,ICUChartDate,Age,Temperature,MeanArterialPressure,HeartRate,RespiratoryRate,FiO2,...,AdmissionDate,UNIT_ID,CREATEDON,RNK,APACHE_WARD,CUSTOMERSTATUS,LOCATIONID,LOCATION,DISCHARGEDATE,PERIOD_WID
0,4978,APD1.0012042887,DELIP573294,1/23/26 0:00,83,36.9,103.00,100.0,20.0,40.0,...,1/21/26 0:00,4,2026-02-10T01:36:11.1029417Z,1,6th Flr T2,ALIVE,10701.0,Delhi - Sarita Vihar,10:00.4,20260123
1,10124,AHS.0000417587,AHSIP101940,02-04-2026 00:00,44,36.0,84.00,78.0,20.0,30.0,...,02-03-2026 00:00,8,2026-02-10T01:38:57.3490126Z,1,GENERAL WARD LEVEL 5,ALIVE,10391.0,Apollo hospital Sheshadripuram,NaN,20260204
2,36594,AHLG.000582499,AHLGIP68174,02-03-2026 00:00,75,36.7,73.26,104.0,20.0,30.0,...,02-02-2026 00:00,1,2026-02-10T01:33:13.9669818Z,1,NEURO ICU,ALIVE,17001.0,Assam Hospitals Limited – Guwahati,NaN,20260203
3,4815,ANM1.0001123066,ANMIP185974,1/19/26 0:00,43,37.0,77.00,110.0,16.0,40.0,...,1/19/26 0:00,5,2026-02-10T01:38:05.9903312Z,1,D WARD,DEAD,10551.0,Navi Mumbai,NaN,20260119
4,10027,AHS.0000416624,AHSIP101703,1/25/26 0:00,51,36.0,98.00,96.0,20.0,30.0,...,1/25/26 0:00,8,2026-02-10T01:38:57.3490126Z,1,EMERGENCY,ALIVE,10391.0,Apollo hospital Sheshadripuram,01:18.2,20260125


In [4]:
# ============================================
# DATASET INFORMATION
# ============================================

print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 104557 entries, 0 to 104556
Data columns (total 67 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   ID                      104557 non-null  int64  
 1   UHID                    104557 non-null  object 
 2   IPNumber                104557 non-null  object 
 3   ICUChartDate            104557 non-null  object 
 4   Age                     104557 non-null  int64  
 5   Temperature             104557 non-null  float64
 6   MeanArterialPressure    104557 non-null  float64
 7   HeartRate               104557 non-null  float64
 8   RespiratoryRate         104557 non-null  float64
 9   FiO2                    104557 non-null  float64
 10  pO2                     104557 non-null  float64
 11  pCO2                    104557 non-null  float64
 12  ArterialpH              104557 non-null  float64
 13  Sodium                  104557 non-null  float64
 14  UrineOutput         

In [5]:
# ============================================
# CHECK MISSING VALUES
# ============================================

missing_values = df.isnull().sum()

missing_values = missing_values[
    missing_values > 0
].sort_values(ascending=False)

print(missing_values)

UpdatedBy         104557
UpdatedDate       104557
Ward              104557
DISCHARGEDATE      11976
APACHE_WARD         5462
LOCATIONID           963
LOCATION             963
CUSTOMERSTATUS       958
dtype: int64


In [6]:
# ============================================
# CUSTOMER STATUS DISTRIBUTION
# ============================================

print(df["CUSTOMERSTATUS"].value_counts())

CUSTOMERSTATUS
ALIVE           91673
DEAD            11900
BROUGHT DEAD       23
LIVE                2
0                   1
Name: count, dtype: int64


In [7]:
# ============================================
# CLEAN CUSTOMER STATUS TARGET
# ============================================

# Convert to uppercase
df["CUSTOMERSTATUS"] = df["CUSTOMERSTATUS"].astype(str).str.upper()

# Replace inconsistent labels
df["CUSTOMERSTATUS"] = df["CUSTOMERSTATUS"].replace({
    "LIVE": "ALIVE",
    "BROUGHT DEAD": "DEAD",
    "0": np.nan
})

# Remove missing targets
df = df[df["CUSTOMERSTATUS"].notna()]

# Keep only ALIVE and DEAD
df = df[
    df["CUSTOMERSTATUS"].isin(["ALIVE", "DEAD"])
]

# Convert to binary
df["Mortality_Target"] = df["CUSTOMERSTATUS"].map({
    "ALIVE": 0,
    "DEAD": 1
})

print(df["CUSTOMERSTATUS"].value_counts())

print("\nMortality Target Distribution:")
print(df["Mortality_Target"].value_counts())

CUSTOMERSTATUS
ALIVE    91675
DEAD     11923
Name: count, dtype: int64

Mortality Target Distribution:
Mortality_Target
0    91675
1    11923
Name: count, dtype: int64


In [8]:
# ============================================
# CONVERT DATES PROPERLY
# ============================================

date_columns = [
    "ICUChartDate",
    "AdmissionDate",
    "DOB"
]

for col in date_columns:
    df[col] = pd.to_datetime(
        df[col],
        format="mixed",
        errors="coerce"
    )

print("Date Conversion Completed")

Date Conversion Completed


In [9]:
# ============================================
# CREATE LOS HOURS
# ============================================

df["LOS_Hours"] = (
    df["ICUChartDate"] - df["AdmissionDate"]
).dt.total_seconds() / 3600

print(df["LOS_Hours"].describe())

count    103598.000000
mean         31.683160
std         336.915012
min           0.000000
25%           0.000000
50%          24.000000
75%          24.000000
max       72840.000000
Name: LOS_Hours, dtype: float64


In [10]:
# ============================================
# REMOVE INVALID LOS VALUES
# ============================================

# Remove negative LOS
df = df[df["LOS_Hours"] >= 0]

# Remove extreme ICU outliers
df = df[df["LOS_Hours"] <= 720]

print("Filtered Dataset Shape:", df.shape)

print("\nLOS Summary:")
print(df["LOS_Hours"].describe())

Filtered Dataset Shape: (103437, 69)

LOS Summary:
count    103437.000000
mean         26.654833
std          50.006311
min           0.000000
25%           0.000000
50%          24.000000
75%          24.000000
max         720.000000
Name: LOS_Hours, dtype: float64


In [11]:
# ============================================
# CREATE LOS CATEGORY
# ============================================

def categorize_los(hours):

    if hours < 24:
        return 0      # Short Stay

    elif hours <= 72:
        return 1      # Medium Stay

    else:
        return 2      # Long Stay


df["LOS_Category"] = df["LOS_Hours"].apply(categorize_los)

print(df["LOS_Category"].value_counts())

LOS_Category
0    49354
1    47006
2     7077
Name: count, dtype: int64


In [12]:
# ============================================
# REMOVE LEAKAGE + USELESS COLUMNS
# ============================================

drop_columns = [

    # IDs
    "ID",
    "UHID",
    "IPNumber",

    # Dates
    "ICUChartDate",
    "AdmissionDate",
    "DOB",
    "CreatedDate",
    "UpdatedDate",
    "CREATEDON",
    "DISCHARGEDATE",

    # Metadata
    "CreatedBy",
    "UpdatedBy",
    "Ward",
    "AdmittingDoctor",
    "LOCATION",
    "LOCATIONID",
    "UNIT_ID",
    "PERIOD_WID",
    "RNK",

    # Original target
    "CUSTOMERSTATUS"
]

df.drop(columns=drop_columns, inplace=True, errors="ignore")

print("Remaining Columns:", df.shape[1])

print(df.columns.tolist())

Remaining Columns: 50
['Age', 'Temperature', 'MeanArterialPressure', 'HeartRate', 'RespiratoryRate', 'FiO2', 'pO2', 'pCO2', 'ArterialpH', 'Sodium', 'UrineOutput', 'Creatinine', 'Urea', 'BSL', 'Albumin', 'Bilirubin', 'Hematocrit', 'WBC', 'IsGCSNotAvailable', 'GCSEyes', 'GCSVerbal', 'GCSMotor', 'MecanicalVentilation', 'CRF', 'Lymphoma', 'Cirrhosis', 'Leukemia', 'HepaticFailure', 'Immunosuppression', 'MetastaticCarcinoma', 'AIDS', 'PreICULengthOfStay', 'DiagnosisType', 'Origin', 'EmergencySurgery', 'Readmission', 'Thrombolysis', 'RespiratoryQuotient', 'AtmosphericPressure', 'SystemValue', 'DiagnosisValue', 'Gender', 'ApacheivScore', 'ApsScore', 'EstimatedMortalityRate', 'EstimatedLengthOfStay', 'APACHE_WARD', 'Mortality_Target', 'LOS_Hours', 'LOS_Category']


In [13]:
# ============================================
# CHECK CATEGORICAL COLUMNS
# ============================================

categorical_cols = df.select_dtypes(include="object").columns

print("Categorical Columns:\n")

for col in categorical_cols:
    print(f"\n{col}")
    print(df[col].value_counts().head())

Categorical Columns:


MecanicalVentilation
MecanicalVentilation
a    86188
b    17249
Name: count, dtype: int64

SystemValue
SystemValue
Cardiovascular    27793
Respiratory       13667
Neurologic        12498
Digestive          9263
Sepsis             8273
Name: count, dtype: int64

DiagnosisValue
DiagnosisValue
Other            19027
Diagnosis        14892
General/Other     5566
Renal/Other       4750
Stroke            4550
Name: count, dtype: int64

Gender
Gender
Male      64993
Female    38442
Others        2
Name: count, dtype: int64

APACHE_WARD
APACHE_WARD
EMERGENCY         16048
Emergency Ward    10714
EMERGENCY WARD     8261
ICU                7199
CICU               4189
Name: count, dtype: int64


In [14]:
# ============================================
# CLEAN CATEGORICAL FEATURES
# ============================================

# ----------------------------
# Mechanical Ventilation
# ----------------------------

df["MecanicalVentilation"] = (
    df["MecanicalVentilation"]
    .replace({
        "a": 0,
        "b": 1
    })
)

# ----------------------------
# Gender
# ----------------------------

df["Gender"] = (
    df["Gender"]
    .replace({
        "Male": 0,
        "Female": 1,
        "Others": 2
    })
)

# ----------------------------
# APACHE WARD CLEANING
# ----------------------------

df["APACHE_WARD"] = (
    df["APACHE_WARD"]
    .astype(str)
    .str.upper()
    .str.strip()
)

# Normalize emergency naming
df["APACHE_WARD"] = (
    df["APACHE_WARD"]
    .replace({
        "EMERGENCY WARD": "EMERGENCY",
        "EMERGENCYWARD": "EMERGENCY"
    })
)

print("Categorical Cleaning Completed")

Categorical Cleaning Completed


In [15]:
# ============================================
# LABEL ENCODING
# ============================================

from sklearn.preprocessing import LabelEncoder

label_encoders = {}

categorical_cols = [
    "SystemValue",
    "DiagnosisValue",
    "APACHE_WARD"
]

for col in categorical_cols:

    le = LabelEncoder()

    df[col] = le.fit_transform(
        df[col].astype(str)
    )

    label_encoders[col] = le

print("Encoding Completed")

print("\nRemaining Data Types:\n")
print(df.dtypes.value_counts())

Encoding Completed

Remaining Data Types:

int64      28
float64    22
Name: count, dtype: int64


In [16]:
# ============================================
# CHECK REMAINING MISSING VALUES
# ============================================

missing_values = df.isnull().sum()

missing_values = missing_values[
    missing_values > 0
].sort_values(ascending=False)

print(missing_values)

Series([], dtype: int64)


In [17]:
# ============================================
# HANDLE MISSING VALUES
# ============================================

# Numerical columns
numerical_cols = df.select_dtypes(
    include=["int64", "float64"]
).columns

# Fill missing numerical values with median
for col in numerical_cols:

    df[col] = df[col].fillna(
        df[col].median()
    )

print("Missing Values After Cleaning:\n")

print(df.isnull().sum().sum())

Missing Values After Cleaning:

0


In [18]:
# ============================================
# CREATE FINAL DATASETS
# ============================================

# ----------------------------
# Mortality Dataset
# ----------------------------

mortality_df = df.drop(
    columns=["LOS_Hours", "LOS_Category"]
)

# ----------------------------
# LOS Regression Dataset
# ----------------------------

los_regression_df = df.drop(
    columns=["Mortality_Target", "LOS_Category"]
)

# ----------------------------
# LOS Classification Dataset
# ----------------------------

los_classification_df = df.drop(
    columns=["Mortality_Target", "LOS_Hours"]
)

print("Mortality Dataset Shape:", mortality_df.shape)

print("LOS Regression Dataset Shape:", los_regression_df.shape)

print("LOS Classification Dataset Shape:", los_classification_df.shape)

Mortality Dataset Shape: (103437, 48)
LOS Regression Dataset Shape: (103437, 48)
LOS Classification Dataset Shape: (103437, 48)


In [19]:
# ============================================
# SAVE PROCESSED DATASETS
# ============================================

mortality_df.to_csv(
    "../data/processed/mortality_processed.csv",
    index=False
)

los_regression_df.to_csv(
    "../data/processed/los_regression_processed.csv",
    index=False
)

los_classification_df.to_csv(
    "../data/processed/los_classification_processed.csv",
    index=False
)

print("Processed Datasets Saved Successfully")

Processed Datasets Saved Successfully
